# Train Mingat on a Colab GPU

This notebook trains three models on your books, one after another, and scores each one on the six
test books that no model trains on. The best model so far, trained on a CPU, scores **1.018** bits per
byte there (lower is better).

| Run | Model | Passes over the books | Question |
|---|---|---|---|
| `micro_6ep` | micro, 6.3M parameters | 6 | The recipe of the 1.018 model on 2.3× the books: what do the new books alone give? |
| `small_6ep` | small, 19.7M parameters | 6 | Does a 3× bigger model help? |
| `small_12ep` | small | up to 12 | Do more passes help the bigger model? (Stops early if it only memorizes.) |

**Before you start**

1. Upload `mingat_books.zip` to Google Drive, into a folder named `mingat`, so that it is at
   `My Drive/mingat/mingat_books.zip`. (`python prepare.py --zip data/mingat_books.zip` makes it.)
   The books stay private in your Drive; nothing is uploaded anywhere else.
2. **Runtime → Change runtime type → T4 GPU → Save.**
3. **Runtime → Run all.** Allow access to Google Drive when asked.

**How long:** the smoke check (step 5) measures the GPU and prints an estimate. Expect 1 to 3 hours.
Keep this tab open; Colab may stop a session that nobody watches.

**If Colab disconnects or stops:** Runtime → Run all again. Finished runs are skipped, and an
interrupted run continues from its last checkpoint, which is saved to your Drive every 10 minutes.

**When it is done:** download the folder `My Drive/mingat/runs` (right-click → Download) and unzip it
into the repo's `out/` folder, so that you have `out/runs/micro_6ep/model.pt` and so on.

In [ ]:
# 1. The GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all."
print("PyTorch", torch.__version__, "|", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

In [ ]:
# 2. Google Drive: the books come from there, and every run is saved there
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/mingat"
BOOKS_ZIP = f"{DRIVE}/mingat_books.zip"
RUNS = f"{DRIVE}/runs"
assert os.path.exists(BOOKS_ZIP), f"Upload mingat_books.zip to Google Drive first, so that it is at {BOOKS_ZIP}"
print("books:", BOOKS_ZIP, "| runs go to:", RUNS)

In [ ]:
# 3. The code, from GitHub
import subprocess

def sh(command):
    # Run a shell command and show its output as it comes. If it fails, stop here (so "Run all" stops too).
    p = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"failed (exit code {p.returncode}): {command}")

%cd /content
sh("test -d mingat || git clone -q https://github.com/theTulgaa/mingat.git")
%cd /content/mingat
sh("git pull -q --ff-only && git log -1 --format='code: %h %s'")

In [ ]:
# 4. The books: train/ (103 books) and test/ (the 6 test books)
sh(f'rm -rf /content/books && mkdir -p /content/books && unzip -q "{BOOKS_ZIP}" -d /content/books')
sh('echo "training books: $(ls /content/books/train | wc -l), test books: $(ls /content/books/test | wc -l)"; du -sh /content/books/*')

In [ ]:
# 5. Smoke check, a few minutes: 300 steps of `small`, paused and resumed, then the speed and an estimate
sh(f'python -u sweep.py --data /content/books/train --test /content/books/test --out "{RUNS}" --smoke')

In [ ]:
# 6. The three runs. Each prints a line every 500 steps: validation loss and speed. Run all again after a disconnect.
sh(f'python -u sweep.py --data /content/books/train --test /content/books/test --out "{RUNS}"')

In [ ]:
# 7. What each model writes after "Тэр өдөр" (the same seed for all)
import glob
for model in sorted(glob.glob(f"{RUNS}/*/model.pt")):
    print("==", model.split("/")[-2])
    sh(f'python generate.py --model "{model}" --prompt "Тэр өдөр" --tokens 100 --seed 0')

**Done.** Download `My Drive/mingat/runs` (right-click → Download), unzip it into the repo's `out/`
folder, and say so: the models get compared on the test books, word by word and sentence by sentence,
on your PC.